In [1]:
import os
os.environ["FIFTYONE_API_URI"] = "https://reverse-fashion-api.fiftyone.ai"
os.environ["FIFTYONE_API_KEY"] = "6a845df30ffb0e164ebe93a1|phrOiupiuEItQXfFT1v8KO0zN3Buobqim2IwZUzcJRI"

In [2]:
import fiftyone as fo

/home/sagemaker-user/reverse-fashion-gender-season--training/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
dataset = fo.load_dataset("sellpy2")
sample = dataset.first()
print("Filepath:", sample.filepath)
print("Media type:", dataset.media_type)
print("Group slices:", dataset.group_slices)

Filepath: s3://reversefashion-images/s/QJs0CvdxmU-NMke6Vkru8-d85c-single.jpg
Media type: group
Group slices: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', 'flatlay']


# ResNet-50 Classifier Training on FiftyOne Sellpy Dataset

Images are stored on S3 (`s3://reversefashion-images/`) and accessible directly from this SageMaker instance — no download needed.

**Configuration:** Set `TARGET_FIELD` to the FiftyOne field path you want to classify (e.g. `"demography"`, `"season"`, `"category_lvl0"`).

In [ ]:
# === CONFIGURATION ===
TARGET_FIELD = "demography"  # FiftyOne field path to classify — change this to your target
BATCH_SIZE = 64
NUM_EPOCHS = 10
LEARNING_RATE = 1e-4
IMAGE_SIZE = 224
VAL_SPLIT = 0.1
GROUP_SLICE = "0"  # Which image slice to use from the group dataset

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from collections import Counter
from helper import build_dataloaders, build_resnet50, train

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# Load dataset and filter to samples that have the target field populated
view = dataset.match(fo.ViewField(TARGET_FIELD).exists())
view = view.select_group_slices(GROUP_SLICE)
print(f"Samples with '{TARGET_FIELD}' populated: {len(view)}")

# Build class mapping
labels = view.values(TARGET_FIELD)
unique_labels = sorted(set(l for l in labels if l is not None))
class_to_idx = {label: idx for idx, label in enumerate(unique_labels)}
num_classes = len(unique_labels)
print(f"Number of classes: {num_classes}")
print(f"Classes: {unique_labels}")
print(f"Class distribution: {Counter(labels).most_common(10)}")

In [ ]:
# Bulk fetch filepaths and labels (fast server-side operation)
filepaths = view.values("filepath")
target_values = view.values(TARGET_FIELD)
sample_data = [(fp, class_to_idx[lbl]) for fp, lbl in zip(filepaths, target_values) if lbl in class_to_idx]
print(f"Total usable samples: {len(sample_data)}")

In [ ]:
train_loader, val_loader, train_size, val_size = build_dataloaders(
    sample_data, VAL_SPLIT, IMAGE_SIZE, BATCH_SIZE,
)
print(f"Train samples: {train_size}, Val samples: {val_size}")

In [ ]:
model = build_resnet50(num_classes, device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)
print(f"Model: ResNet-50 | Output classes: {num_classes}")

In [ ]:
train(model, train_loader, val_loader, criterion, optimizer, scheduler,
     device, NUM_EPOCHS, class_to_idx, TARGET_FIELD)